Cell 1: Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit

# Snowflake session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Set random seed for reproducibility
RANDOM_STATE = 42

print("Libraries loaded successfully!")
print(f"Random seed: {RANDOM_STATE}")

Cell 2: Load Raw Water Quality Data

In [ ]:
# Load the complete training dataset (9,319 samples)
df = pd.read_csv('water_quality_training_dataset.csv')

print(f"Data loaded: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Cell 3: Data Inspection

In [ ]:
# Basic statistics
print("="*80)
print("DATA SUMMARY")
print("="*80)

print(f"\nTotal samples: {len(df):,}")

# Create location identifier from lat/lon
df['Location_ID'] = df.groupby(['Latitude', 'Longitude']).ngroup()
print(f"Unique locations: {df['Location_ID'].nunique()}")

# Date range
df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True)
print(f"\nDate range: {df['Sample Date'].min()} to {df['Sample Date'].max()}")

# Samples per location
print(f"\nSamples per location:")
print(f"  Mean: {df.groupby('Location_ID').size().mean():.1f}")
print(f"  Median: {df.groupby('Location_ID').size().median():.1f}")
print(f"  Min: {df.groupby('Location_ID').size().min()}")
print(f"  Max: {df.groupby('Location_ID').size().max()}")

# Target statistics
print(f"\nTarget variable summary:")
for target in ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']:
    print(f"\n{target}:")
    print(f"  Mean: {df[target].mean():.2f}")
    print(f"  Median: {df[target].median():.2f}")
    print(f"  Std: {df[target].std():.2f}")
    print(f"  Min: {df[target].min():.2f}")
    print(f"  Max: {df[target].max():.2f}")
    print(f"  Missing: {df[target].isna().sum()}")


Cell 4: Create 70/10/20 Location-Stratified Split

In [ ]:
# Create location groups for splitting
# Each unique (lat, lon) pair is a location
location_groups = df['Location_ID']

print("="*80)
print("CREATING TRAIN/VAL/TEST SPLIT (70/10/20)")
print("="*80)

# First split: 80% development (train+val) / 20% test
splitter1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)

for dev_idx, test_idx in splitter1.split(df, groups=location_groups):
    df_dev = df.iloc[dev_idx].copy()
    df_test = df.iloc[test_idx].copy()

print(f"\nFirst split (dev/test):")
print(f"  Development set: {len(df_dev):,} samples ({len(df_dev)/len(df)*100:.1f}%)")
print(f"  Test set: {len(df_test):,} samples ({len(df_test)/len(df)*100:.1f}%)")
print(f"  Dev locations: {df_dev['Location_ID'].nunique()}")
print(f"  Test locations: {df_test['Location_ID'].nunique()}")

# Second split: Development → 87.5% train / 12.5% val (to get 70/10 of original)
# 87.5% of 80% = 70% of original
# 12.5% of 80% = 10% of original
splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=RANDOM_STATE+1)
dev_groups = df_dev['Location_ID']

for train_idx, val_idx in splitter2.split(df_dev, groups=dev_groups):
    df_train = df_dev.iloc[train_idx].copy()
    df_val = df_dev.iloc[val_idx].copy()

print(f"\nSecond split (train/val):")
print(f"  Train set: {len(df_train):,} samples ({len(df_train)/len(df)*100:.1f}%)")
print(f"  Val set: {len(df_val):,} samples ({len(df_val)/len(df)*100:.1f}%)")
print(f"  Train locations: {df_train['Location_ID'].nunique()}")
print(f"  Val locations: {df_val['Location_ID'].nunique()}")

print(f"\n{'='*80}")
print("FINAL SPLIT SUMMARY")
print(f"{'='*80}")
print(f"  Train: {len(df_train):,} samples ({len(df_train)/len(df)*100:.1f}%) | {df_train['Location_ID'].nunique()} locations")
print(f"  Val:   {len(df_val):,} samples ({len(df_val)/len(df)*100:.1f}%) | {df_val['Location_ID'].nunique()} locations")
print(f"  Test:  {len(df_test):,} samples ({len(df_test)/len(df)*100:.1f}%) | {df_test['Location_ID'].nunique()} locations")
print(f"  Total: {len(df):,} samples | {df['Location_ID'].nunique()} locations")

Cell 5: Verify No Location Overlap

In [ ]:
# Critical check: Ensure no locations appear in multiple splits
train_locs = set(df_train['Location_ID'].unique())
val_locs = set(df_val['Location_ID'].unique())
test_locs = set(df_test['Location_ID'].unique())

train_val_overlap = train_locs & val_locs
train_test_overlap = train_locs & test_locs
val_test_overlap = val_locs & test_locs

print("="*80)
print("LOCATION OVERLAP CHECK")
print("="*80)
print(f"Train ∩ Val: {len(train_val_overlap)} locations (should be 0)")
print(f"Train ∩ Test: {len(train_test_overlap)} locations (should be 0)")
print(f"Val ∩ Test: {len(val_test_overlap)} locations (should be 0)")

if len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
    print("\nSUCCESS: No location overlap between splits!")
else:
    print("\nERROR: Location overlap detected! Check splitting logic.")

Cell 6: Target Distribution Comparison

In [ ]:
# Verify target distributions are similar across splits
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

targets = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

for i, target in enumerate(targets):
    axes[i].hist(df_train[target].dropna(), bins=50, alpha=0.5, label='Train', density=True)
    axes[i].hist(df_val[target].dropna(), bins=30, alpha=0.5, label='Val', density=True)
    axes[i].hist(df_test[target].dropna(), bins=30, alpha=0.5, label='Test', density=True)
    axes[i].set_xlabel(target)
    axes[i].set_ylabel('Density')
    axes[i].set_title(f'{target}\nDistribution Across Splits')
    axes[i].legend()

plt.tight_layout()
plt.savefig('split_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("Target distribution plots saved as 'split_distributions.png'")

Cell 7: Geographic Distribution Check

In [ ]:
# Visualize geographic distribution of splits
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

ax.scatter(df_train['Longitude'], df_train['Latitude'], 
           c='blue', s=20, alpha=0.3, label=f'Train ({len(df_train)} samples)')
ax.scatter(df_val['Longitude'], df_val['Latitude'], 
           c='orange', s=20, alpha=0.5, label=f'Val ({len(df_val)} samples)')
ax.scatter(df_test['Longitude'], df_test['Latitude'], 
           c='red', s=20, alpha=0.7, label=f'Test ({len(df_test)} samples)')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Geographic Distribution of Train/Val/Test Splits')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('split_geography.png', dpi=150, bbox_inches='tight')
plt.show()

print("Geographic distribution plot saved as 'split_geography.png'")

Cell 8: Save Split Files

In [ ]:
import base64
from IPython.display import HTML, display

# Save the three splits
print("="*80)
print("SAVING SPLIT FILES")
print("="*80)

# Save to CSV (for use in next notebooks)
df_train.to_csv('train70.csv', index=False)
df_val.to_csv('val10.csv', index=False)
df_test.to_csv('test20.csv', index=False)

print(f"\nSaved: train70.csv {df_train.shape}")
print(f"Saved: val10.csv {df_val.shape}")
print(f"Saved: test20.csv {df_test.shape}")

# Generate download links
print("\n" + "="*80)
print("DOWNLOAD LINKS")
print("="*80)

for fname, df_split in [("train70.csv", df_train),
                        ("val10.csv", df_val),
                        ("test20.csv", df_test)]:
    csv = df_split.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    display(HTML(f'<a href="data:file/csv;base64,{b64}" '
                f'download="{fname}">Download {fname}</a>'))

Cell 9: Create Unique Extraction Lists

In [ ]:
# Create lists of unique (lat, lon, date) combinations for feature extraction
# This is what you'll use in the next notebooks

# Combine all three splits to get ALL unique combinations
df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

# Unique (lat, lon, date) for time-varying features (TerraClimate, Landsat)
unique_samples = df_all[['Latitude', 'Longitude', 'Sample Date']].drop_duplicates()
unique_samples = unique_samples.sort_values(['Latitude', 'Longitude', 'Sample Date']).reset_index(drop=True)

print("="*80)
print("UNIQUE EXTRACTION LISTS")
print("="*80)
print(f"\nUnique (lat, lon, date) combinations: {len(unique_samples):,}")
print("  (Use this for TerraClimate and Landsat extraction)")

# Unique (lat, lon) for static features (Elevation, Land Cover)
unique_locations = df_all[['Latitude', 'Longitude']].drop_duplicates()
unique_locations = unique_locations.sort_values(['Latitude', 'Longitude']).reset_index(drop=True)

print(f"\nUnique (lat, lon) locations: {len(unique_locations):,}")
print("  (Use this for Elevation and Land Cover extraction)")

# Save these for reference in next notebooks
unique_samples.to_csv('unique_samples_for_extraction.csv', index=False)
unique_locations.to_csv('unique_locations_for_extraction.csv', index=False)

print(f"\n✅ Saved: unique_samples_for_extraction.csv")
print(f"✅ Saved: unique_locations_for_extraction.csv")

# Generate download links
print("\n" + "="*80)
print("DOWNLOAD LINKS")
print("="*80)

for fname, df_unique in [("unique_samples_for_extraction.csv", unique_samples),
                         ("unique_locations_for_extraction.csv", unique_locations)]:
    csv = df_unique.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    display(HTML(f'<a href="data:file/csv;base64,{b64}" '
                f'download="{fname}">Download {fname}</a>'))

Cell 10: Final Summary & Next Steps

In [ ]:
print("="*80)
print("NOTEBOOK 1 COMPLETE: DATA SPLITTING")
print("="*80)

print("\n📊 WHAT YOU HAVE NOW:")
print("  ✅ train70.csv - Training set (70% of data)")
print("  ✅ val10.csv - Validation set (10% of data)")
print("  ✅ test20.csv - Test set (20% of data)")
print("  ✅ unique_samples_for_extraction.csv - For TerraClimate/Landsat")
print("  ✅ unique_locations_for_extraction.csv - For Elevation/Land Cover")

print("\n🎯 NEXT STEPS:")
print("  1. Notebook 2: Extract TerraClimate features")
print("  2. Notebook 3: Extract Landsat features")
print("  3. Notebook 4: Extract Elevation features")
print("  4. Notebook 5: Extract Land Cover features")
print("  5. Notebook 6: Feature engineering & merging")

print("\n⚠️  IMPORTANT:")
print("  - Do NOT modify train70.csv, val10.csv, or test20.csv")
print("  - Do NOT touch test20.csv during model development")
print("  - Use train70 for training, val10 for tuning/evaluation")
print("  - Save test20 for final evaluation ONCE at the end")

print("\n" + "="*80)
print("Ready to proceed to Notebook 2: TerraClimate Extraction!")
print("="*80)